# Neuron Neighborhood Atlas
#### Compares brain two brain regions which surround a chosen target cell type in brain tissue.
#### Demo will look at doapmine neurons in the substantia nigra pars compacta (SNc) verses dopamine neurons in the Ventral Tegmental Area (VTA).

In [10]:
# import libraries
import argparse
import warnings
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import false_discovery_control

## Parser Arguments

In [12]:
parser = argparse.ArgumentParser(description="Measure cell neighborhoods around a target cell type")
parser.add_argument("--data", default="data/midbrain_cells.parquet",
                    help="Parquet file is made by prepare_data.py")
parser.add_argument("--regions", nargs="+", default=["SNc", "VTA"],
                    help="CCF structures for analysis (two regions requied for the comparison test)")
parser.add_argument("--target-value", default="Dopa",
                    help="The value in --target-column that marks a target cell")
parser.add_argument("--neighbor-level", default="class", choices=["class", "subclass"],
                    help="Cell-type level used to label neighbors")
parser.add_argument("--radius", type=float, default=50,
                    help="Neighborhood radius in micrometers (default = 50)")
parser.add_argument("--sections", nargs="*", default=None,
                    help="Brain sections to include (default = every section with available targets.)")
parser.add_argument("--permutations", type=int, default=999,
                    help="Number of permutations for signficance test.")
parser.add_argument("--seed", type=int, default=592,
                    help="Random seed for reproducibility.")
parser.add_argument("--outdir", default="outputs",
                    help="Folder containing maps, tables, and GeoPackage.")
args = parser.parse_args([])

outdir = Path(args.outdir)
outdir.mkdir(parents=True, exist_ok=True)
level = args.neighbor_level

## Loading Cells

In [13]:
print("Loading cells...")
cells = pd.read_parquet(args.data)
## coordinates are in mm
cells["x_um"] = cells["x_section"] * 1000  
cells["y_um"] = cells["y_section"] * 1000
cells[level] = cells[level].fillna("Unassigned")
is_target = (
    cells["parcellation_structure"].isin(args.regions)
    & (cells[args.target_column] == args.target_value)
)
sections = args.sections or sorted(cells.loc[is_target, "brain_section_label"].unique())
cells = cells[cells["brain_section_label"].isin(sections)].reset_index(drop=True)
is_target = (
    cells["parcellation_structure"].isin(args.regions)
    & (cells[args.target_column] == args.target_value)
)

gcells = gpd.GeoDataFrame(cells, geometry=gpd.points_from_xy(cells["x_um"], cells["y_um"]))
targets = gcells[is_target].copy()
if targets.empty:
    raise SystemExit("No target cells identified. Check --regions, --target-column and --target-value.")

print(f"  {len(gcells):,} cells in {len(sections)} sections")
print("  Target cells per region:", targets["parcellation_structure"].value_counts().to_dict())


Loading cells...


AttributeError: 'Namespace' object has no attribute 'target_column'

## Buffer and Spatial Analysis for Neighborhoods

In [4]:
print(f"Finding neighbors within {args.radius:g} µm...")
buffers = targets[["cell_label", "brain_section_label", "parcellation_structure", "geometry"]].copy()
buffers["geometry"] = buffers.buffer(args.radius)
buffers = buffers.rename(columns={"cell_label": "target_label",
                                  "brain_section_label": "target_section",
                                  "parcellation_structure": "target_region"})

pairs = gpd.sjoin(gcells[["cell_label", "brain_section_label", level, "geometry"]],
                  buffers, predicate="within")
pairs = pairs[(pairs["brain_section_label"] == pairs["target_section"])
              & (pairs["cell_label"] != pairs["target_label"])]

counts = (pairs.groupby(["target_label", level]).size()
          .unstack(fill_value=0)
          .reindex(targets["cell_label"], fill_value=0))
counts.index.name = "cell_label"
total = counts.sum(axis=1)
props = counts.div(total.replace(0, np.nan), axis=0).fillna(0)

region_of = targets.set_index("cell_label")["parcellation_structure"]
section_of = targets.set_index("cell_label")["brain_section_label"]

composition = props.groupby(region_of).mean().T
composition.columns = [f"mean_prop_{r}" for r in composition.columns]
summary = pd.DataFrame({
    "n_targets": region_of.value_counts(),
    "mean_neighbors": total.groupby(region_of).mean(),
})
print("\nNeighborhood summary:\n", summary.round(2).to_string())

NameError: name 'args' is not defined

## Permutations

In [ ]:
results = None
if len(args.regions) == 2:
    a, b = args.regions
    print(f"\nRunning {args.permutations} permutations ({a} vs {b}, shuffled within sections)...")
    rng = np.random.default_rng(args.seed)
    # Skip very rare neighbor types: a handful of stray cells can look "significant" by chance.
    common = counts.columns[counts.sum() >= 20]
    stat_table = props[common].copy()
    stat_table["total_neighbors"] = total
    values = stat_table.to_numpy()
    labels = region_of.reindex(stat_table.index).to_numpy()
    groups = [np.flatnonzero(section_of.reindex(stat_table.index).to_numpy() == s)
              for s in sections]

    def mean_difference(lab):
        return values[lab == a].mean(axis=0) - values[lab == b].mean(axis=0)

    observed = mean_difference(labels)
    exceed = np.zeros_like(observed)
    for _ in range(args.permutations):
        shuffled = labels.copy()
        # shuffle region labels only within each section
        for idx in groups: 
            shuffled[idx] = rng.permutation(shuffled[idx])
        exceed += np.abs(mean_difference(shuffled)) >= np.abs(observed)

    p = (exceed + 1) / (args.permutations + 1)
    results = pd.DataFrame({
        "measure": stat_table.columns,
        f"mean_{a}": values[labels == a].mean(axis=0),
        f"mean_{b}": values[labels == b].mean(axis=0),
        f"difference_{a}_minus_{b}": observed,
        "p_value": p,
        "q_value_fdr": false_discovery_control(p),
    }).sort_values("p_value")
    results["significant"] = results["q_value_fdr"] < 0.05
    print(results.round(4).to_string(index=False))

## Region Outlines

In [5]:
outline_rows = []
for (section, region), grp in gcells[gcells["parcellation_structure"].isin(args.regions)].groupby(["brain_section_label", "parcellatoin_structure"]):
    merged = grp.buffer(40).union_all().buffer(-20)
    for part in getattr(merged, "geoms", [merged]):
        n_inside = int(grp.within(part).sum())
        if n_inside >= 10:
            outline_rows.append({"brain_section_label": section, "region": region, "n_cells": n_inside, "geometry": part})
outlines = gpd.GeoDataFrame(outline_rows, geometry="geometry")

NameError: name 'gcells' is not defined

## Maps (1 PDF page per section)


In [6]:
categories = props.mean().sort_values(ascending=False).index.tolist()
palette = plt.get_cmap("tab10").colors
colors = {c: palette[i % 10] for i, c in enumerate(categories)}
region_styles = {r: s for r, s in zip(args.regions, ["-", "--", ":", "-."])}

def draw_section(ax, section):
    sec = gcells[gcells["brain_section_label"] == section]
    local = sec[sec["parcellation_structure"].isin(args.regions)]
    sec_targets = targets[targets["brain_section_label"] == section]
    xmin, ymin, xmax, ymax = local.total_bounds
    pad = 400
    view = sec.cx[xmin - pad:xmax + pad, ymin - pad:ymax + pad]

    ax.scatter(view["x_um"], view["y_um"], s=2, color="0.85", linewidths=0)
    for cat, grp in view[view["parcellation_structure"].isin(args.regions)].groupby(level):
        ax.scatter(grp["x_um"], grp["y_um"], s=6, color=colors.get(cat, "0.5"),
                   linewidths=0, label=cat)
    sec_targets.buffer(args.radius).boundary.plot(ax=ax, color="black", linewidth=0.3, alpha=0.4)
    ax.scatter(sec_targets["x_um"], sec_targets["y_um"], s=14, facecolor="none",
               edgecolor="black", linewidths=0.7, label=f"{args.target_value} target")
    for _, row in outlines[outlines["brain_section_label"] == section].iterrows():
        x, y = row.geometry.exterior.xy
        ax.plot(x, y, color="black", linestyle=region_styles[row["region"]], linewidth=1.2)
        minx, miny, maxx, maxy = row.geometry.bounds
        ax.text((minx + maxx) / 2, maxy + 40, row["region"], fontsize=10,
                fontweight="bold", ha="center", va="top")

    # 200 µm scale bar
    sx, sy = xmin - pad + 50, ymax + pad - 80
    ax.plot([sx, sx + 200], [sy, sy], color="black", linewidth=2)
    ax.text(sx + 100, sy - 30, "200 µm", ha="center", fontsize=8)

    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymax + pad, ymin - pad)  # tissue y axis is reverse
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    counts_txt = sec_targets["parcellation_structure"].value_counts().to_dict()
    ax.set_title(f"Section {section.split('.')[-1]}  ·  targets: {counts_txt}", fontsize=10)


print("\nDrawing maps...")
with PdfPages(outdir / "section_maps.pdf") as pdf:
    for section in sections:
        if not (targets["brain_section_label"] == section).any():
            continue
        fig, ax = plt.subplots(figsize=(9, 7))
        draw_section(ax, section)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7,
                  markerscale=2, frameon=False, title=f"Cells in {' / '.join(args.regions)}")
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

best = targets["brain_section_label"].value_counts().idxmax()
fig, ax = plt.subplots(figsize=(9, 7))
draw_section(ax, best)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7, markerscale=2, frameon=False)
fig.tight_layout()
fig.savefig(outdir / "example_section_map.png", dpi=200, bbox_inches="tight")
plt.close(fig)

NameError: name 'props' is not defined

## Composition

In [7]:
significant = set(results.loc[results["significant"], "measure"]) if results is not None else set()
shown = [c for c in categories if composition.loc[c].max() >= 0.02 or c in significant]
fig, ax = plt.subplots(figsize=(8, 0.5 * len(shown) + 1.5))
y = np.arange(len(shown))
height = 0.8 / len(args.regions)
for i, region in enumerate(args.regions):
    vals = composition.loc[shown, f"mean_prop_{region}"]
    ax.barh(y + i * height, vals, height, label=f"{region} (n={summary.loc[region, 'n_targets']})",
            color=["#3D6FB6", "#E07B39", "#5AA469", "#8E6BBF"][i])
if results is not None:
    sig = results.set_index("measure")["significant"]
    for j, cat in enumerate(shown):
        if sig.get(cat, False):
            ax.text(composition.loc[cat].max() + 0.01, j + height / 2, "*", fontsize=14, va="center")
ax.set_yticks(y + height * (len(args.regions) - 1) / 2)
ax.set_yticklabels(shown)
ax.invert_yaxis()
ax.set_xlabel(f"Mean share of neighbors within {args.radius:g} µm")
ax.set_title(f"Who lives near {args.target_value} neurons?  (* FDR q < 0.05)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(outdir / "neighbor_composition.png", dpi=200)
plt.close(fig)

NameError: name 'results' is not defined

## Exporting

In [8]:
target_out = targets[["cell_label", "brain_section_label", "parcellation_structure",
                      "class", "subclass", "x_um", "y_um", "geometry"]].merge(
    counts.add_prefix("n_").reset_index(), on="cell_label")
target_out["n_neighbors_total"] = total.reindex(target_out["cell_label"]).to_numpy()

counts.add_prefix("n_").join(region_of.rename("region")).to_csv(outdir / "neighbor_counts_per_target.csv")
composition.to_csv(outdir / "composition_by_region.csv", index_label="neighbor_type")
summary.to_csv(outdir / "summary_by_region.csv", index_label="region")
if results is not None:
    results.to_csv(outdir / "permutation_results.csv", index=False)

gpkg = outdir / "neighborhoods.gpkg"
target_out.to_file(gpkg, layer="target_cells")
outlines.to_file(gpkg, layer="region_outlines")
gcells[gcells["parcellation_structure"].isin(args.regions)][
    ["cell_label", "brain_section_label", "parcellation_structure", "class", "subclass", "geometry"]
].to_file(gpkg, layer="region_cells")

print(f"\nDone. Outputs saved to {outdir.resolve()}")

NameError: name 'targets' is not defined

# END OF SCRIPT